In [ ]:
from google.colab import drive   # connect colab to google drive for file transferring
drive.mount('/content/drive')

In [ ]:
!pip install biopython

In [ ]:
import os
import numpy as np
import pandas as pd
from pandas import Series, DataFrame
import random

import itertools
import math

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Variable
from torch.utils.data import TensorDataset, Dataset, DataLoader
import torchvision.datasets as datasets
import torchvision.transforms as transforms
import torchvision.models as models
import matplotlib.pyplot as plt
from tqdm import tqdm

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, balanced_accuracy_score, matthews_corrcoef, precision_recall_curve
from tensorflow.keras.models import load_model

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import matplotlib.pyplot as plt

In [ ]:
from sklearn.metrics import (
    roc_curve,
    auc as compute_auc,
    balanced_accuracy_score,
    matthews_corrcoef,
    precision_recall_curve
)

In [ ]:
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_combine/'
data_test_dir = os.path.join(new_model_dir, f'data/')
original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
original_data_df = pd.read_csv(original_data_path)
original_data_df

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc as compute_auc,
    balanced_accuracy_score, matthews_corrcoef, precision_recall_curve,
    classification_report
)

base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/'
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_combine/'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_combine/'
os.makedirs(save_dir, exist_ok=True)

all_metrics = []

for fold in range(1, 6):
    print(f"\n========== Processing fold {fold} ==========")
    try:
        # 1. Load the trained model
        model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
        model_path = os.path.join(model_dir, 'classification_model_with_val.h5')
        model = load_model(model_path)
        print(f"[OK] Model for fold {fold} loaded.")

        # 2. Load and filter data
        data_test_dir = os.path.join(new_model_dir, 'data/')
        original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
        original_data_df = pd.read_csv(original_data_path)

        merged_df = original_data_df[original_data_df['SaSaRate'] >= 0.2].copy()

        # 3. Predict
        X_test = merged_df[[f'embedding_{i}' for i in range(320)]].to_numpy()
        predictions = model.predict(X_test)
        merged_df['Predicted_Status'] = predictions

        # 4. Clean and normalize
        merged_df['Predicted_Status'] = pd.to_numeric(merged_df['Predicted_Status'], errors='coerce')

        def normalize_group(x):
            if x.isnull().all() or (x.max() == x.min()):
                return np.nan
            else:
                return (x - x.min()) / (x.max() - x.min())

        merged_df['Normalized_Predicted_Status'] = merged_df.groupby('name')['Predicted_Status'].transform(normalize_group)

        # 5. Drop invalid rows
        merged_df = merged_df.replace([np.inf, -np.inf], np.nan)
        merged_df = merged_df.dropna(subset=['Normalized_Predicted_Status', 'Predicted_Status', 'Status'])
        merged_df = merged_df[merged_df['SaSaRate'] >= 0.1].copy()

        # 6. Classification metrics
        y_true = merged_df['Status'].replace({'Interacting': 1, 'Non-Interacting': 0}).astype(int)
        y_pred = (merged_df['Predicted_Status'] >= 0.5).astype(int)
        y_scores = merged_df['Normalized_Predicted_Status']

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        auc_val = roc_auc_score(y_true, y_scores)

        print("Classification Report:")
        print(classification_report(y_true, y_pred))
        print(f"Accuracy:  {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall:    {rec:.4f}")
        print(f"F1 Score:  {f1:.4f}")
        print(f"AUC:       {auc_val:.4f}")

        # 7. Confusion matrix
        conf_matrix = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 6))
        plt.imshow(conf_matrix, cmap="Blues", interpolation="nearest")
        plt.title(f"Confusion Matrix (Fold {fold})")
        plt.colorbar()
        plt.xticks([0, 1], ['Non-Interacting', 'Interacting'])
        plt.yticks([0, 1], ['Non-Interacting', 'Interacting'])
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        for i in range(2):
            for j in range(2):
                plt.text(j, i, conf_matrix[i, j], ha="center", va="center", color="red")
        plt.show()

        # 8. Extra metrics
        fpr, tpr, _ = roc_curve(y_true, y_pred)
        auc_from_curve = compute_auc(fpr, tpr)
        print(f'ROC-AUC from curve: {auc_from_curve:.2f}')

        y_pred_binary = (merged_df['Normalized_Predicted_Status'] >= 0.5).astype(int)
        balanced_acc = balanced_accuracy_score(y_true, y_pred_binary)
        print(f'Balanced Accuracy (BAC): {balanced_acc:.2f}')

        mcc_score = matthews_corrcoef(y_true, y_pred_binary)
        print(f'MCC: {mcc_score:.2f}')

        pr_precision, pr_recall, _ = precision_recall_curve(y_true, y_pred_binary)
        pr_auc_score = compute_auc(pr_recall, pr_precision)
        print(f'PR-AUC: {pr_auc_score:.2f}')

        # 9. Nantigens and Fpredicted
        num_unique_antigens = merged_df['name'].nunique()
        print(f'Nantigens: {num_unique_antigens}')

        fraction_predicted = np.mean(y_pred_binary) * 100
        print(f'Fpredicted: {fraction_predicted:.2f}%')

        all_metrics.append({
        'Fold': fold,
        'AUC': auc_val,
        'F1': f1,
        'MCC': mcc_score,
        'PR-AUC': pr_auc_score,
        'Balanced_Accuracy': balanced_acc,
        'Nantigens': num_unique_antigens,
        'Fpredicted': fraction_predicted
        })

    except Exception as e:
        print(f"[Error in fold {fold}] {str(e)}")

In [ ]:
if all_metrics:
    metrics_df = pd.DataFrame(all_metrics)
    print("\n========== 汇总指标 (5-fold) ==========")
    print(metrics_df.round(4))

    mean_metrics = metrics_df.mean(numeric_only=True)
    std_metrics = metrics_df.std(numeric_only=True)

    print("\n====== 各指标平均值 (mean) ======")
    print(mean_metrics.round(4))

    print("\n====== 各指标标准差 (std) ======")
    print(std_metrics.round(4))
else:
    print("[!] 没有成功运行的 fold，无法汇总指标。")

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc as compute_auc,
    balanced_accuracy_score, matthews_corrcoef, precision_recall_curve,
    classification_report
)

base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/'
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_combine/'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_combine/'
os.makedirs(save_dir, exist_ok=True)

for fold in range(2, 3):
    print(f"\n========== Processing fold {fold} ==========")
    try:
        # 1. Load the trained model
        model_dir = os.path.join(base_model_dir, f'fold_{fold}/data_processed_step1_with_SI/')
        model_path = os.path.join(model_dir, 'classification_model_with_val.h5')
        model = load_model(model_path)
        print(f"[OK] Model for fold {fold} loaded.")

        # 2. Load and filter data
        data_test_dir = os.path.join(new_model_dir, 'data/')
        original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
        original_data_df = pd.read_csv(original_data_path)

        merged_df = original_data_df[original_data_df['SaSaRate'] >= 0.1].copy()

        # 3. Predict
        X_test = merged_df[[f'embedding_{i}' for i in range(320)]].to_numpy()
        predictions = model.predict(X_test)
        merged_df['Predicted_Status'] = predictions

        # 4. Clean and normalize
        merged_df['Predicted_Status'] = pd.to_numeric(merged_df['Predicted_Status'], errors='coerce')

        def normalize_group(x):
            if x.isnull().all() or (x.max() == x.min()):
                return np.nan
            else:
                return (x - x.min()) / (x.max() - x.min())

        merged_df['Normalized_Predicted_Status'] = merged_df.groupby('name')['Predicted_Status'].transform(normalize_group)

        # 5. Drop invalid rows
        merged_df = merged_df.replace([np.inf, -np.inf], np.nan)
        merged_df = merged_df.dropna(subset=['Normalized_Predicted_Status', 'Predicted_Status', 'Status'])
        merged_df = merged_df[merged_df['SaSaRate'] >= 0.1].copy()

        # 6. Classification metrics
        y_true = merged_df['Status'].replace({'Interacting': 1, 'Non-Interacting': 0}).astype(int)
        y_pred = (merged_df['Predicted_Status'] >= 0.5).astype(int)
        y_scores = merged_df['Normalized_Predicted_Status']

        acc = accuracy_score(y_true, y_pred)
        prec = precision_score(y_true, y_pred, zero_division=0)
        rec = recall_score(y_true, y_pred, zero_division=0)
        f1 = f1_score(y_true, y_pred, zero_division=0)
        auc_val = roc_auc_score(y_true, y_scores)

        print("Classification Report:")
        print(classification_report(y_true, y_pred))
        print(f"Accuracy:  {acc:.4f}")
        print(f"Precision: {prec:.4f}")
        print(f"Recall:    {rec:.4f}")
        print(f"F1 Score:  {f1:.4f}")
        print(f"AUC:       {auc_val:.4f}")

        # 7. Confusion matrix
        conf_matrix = confusion_matrix(y_true, y_pred)
        plt.figure(figsize=(6, 6))
        plt.imshow(conf_matrix, cmap="Blues", interpolation="nearest")
        plt.title(f"Confusion Matrix (Fold {fold})")
        plt.colorbar()
        plt.xticks([0, 1], ['Non-Interacting', 'Interacting'])
        plt.yticks([0, 1], ['Non-Interacting', 'Interacting'])
        plt.xlabel('Predicted Label')
        plt.ylabel('True Label')
        for i in range(2):
            for j in range(2):
                plt.text(j, i, conf_matrix[i, j], ha="center", va="center", color="red")
        plt.show()

        # 8. Extra metrics
        fpr, tpr, _ = roc_curve(y_true, y_pred)
        auc_from_curve = compute_auc(fpr, tpr)
        print(f'ROC-AUC from curve: {auc_from_curve:.2f}')

        y_pred_binary = (merged_df['Normalized_Predicted_Status'] >= 0.5).astype(int)
        balanced_acc = balanced_accuracy_score(y_true, y_pred_binary)
        print(f'Balanced Accuracy (BAC): {balanced_acc:.2f}')

        mcc_score = matthews_corrcoef(y_true, y_pred_binary)
        print(f'MCC: {mcc_score:.2f}')

        pr_precision, pr_recall, _ = precision_recall_curve(y_true, y_pred_binary)
        pr_auc_score = compute_auc(pr_recall, pr_precision)
        print(f'PR-AUC: {pr_auc_score:.2f}')

        # 9. Nantigens and Fpredicted
        num_unique_antigens = merged_df['name'].nunique()
        print(f'Nantigens: {num_unique_antigens}')

        fraction_predicted = np.mean(y_pred_binary) * 100
        print(f'Fpredicted: {fraction_predicted:.2f}%')

        all_metrics.append({
        'Fold': fold,
        'AUC': auc_val,
        'F1': f1,
        'MCC': mcc_score,
        'PR-AUC': pr_auc_score,
        'Balanced_Accuracy': balanced_acc,
        'Nantigens': num_unique_antigens,
        'Fpredicted': fraction_predicted
        })

    except Exception as e:
        print(f"[Error in fold {fold}] {str(e)}")

In [ ]:
merged_df[merged_df['name'].isin(["7CWO_1_L_H_A"])]

In [ ]:
merged_df['name'].unique()

In [ ]:
import os
import pandas as pd
from Bio.PDB import PDBParser, PDBIO, is_aa

best_names = ['7DR4_1_L_H_K', '7CWO_1_L_H_A', '7BBJ_1_L_H_A', '7C6A_1_L_H_A_B',
       '7VYT_1_L_H_T']

best_df = merged_df[merged_df['name'].isin(best_names)]
best_df['pred'] = (merged_df['Predicted_Status'] >= 0.5).astype(int)
best_df['ResID'] = best_df['ResID'].astype(str)

input_pdb_path = "/content/drive/MyDrive/@antigen_epitope/abdb_new_combine/data/structures/"
output_pdb_path = "/content/drive/MyDrive/@antigen_epitope/abdb_new_combine/results/"
os.makedirs(output_pdb_path, exist_ok=True)

parser = PDBParser(QUIET=True)
io = PDBIO()

for index, row in best_df.iterrows():
    pdb_filename = row['name'] + '.pdb'
    file_found = False

    for x in range(1, 6):
        input_file = input_pdb_path.format(i=x) + pdb_filename
        if os.path.isfile(input_file):
            file_found = True
            break

    if not file_found:
        print(f"{pdb_filename} not found in any fold directory.")
        continue

    structure = parser.get_structure(row['name'], input_file)

    for model in structure:
        for chain in model:
            for residue in chain:
                if not is_aa(residue, standard=True):
                    continue
                res_id_tuple = residue.get_id()
                res_id = f"{res_id_tuple[0]}{res_id_tuple[1]}{res_id_tuple[2]}".strip()
                matched_rows = best_df[(best_df['name'] == row['name']) & (best_df['ResID'] == res_id)]
                if chain.id in ['L', 'H']:
                    pred_value = 0
                elif not matched_rows.empty:
                    pred_value = matched_rows.iloc[0]['pred']
                else:
                    pred_value = 0
                for atom in residue:
                    atom.set_bfactor(pred_value)

    output_file = os.path.join(output_pdb_path, pdb_filename)
    io.set_structure(structure)
    io.save(output_file)
    print(f"Processed and saved {output_file}")

```pymol
spectrum b, rainbow, minimum=0, maximum=1
color magenta, chain H or chain L
set internal_gui, off
```


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tensorflow.keras.models import load_model
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, roc_curve, auc as compute_auc,
    balanced_accuracy_score, matthews_corrcoef, precision_recall_curve,
    classification_report
)

base_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_method/'
new_model_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_combine/'
save_dir = '/content/drive/MyDrive/@antigen_epitope/abdb_new_combine/'
os.makedirs(save_dir, exist_ok=True)

# 1. Load the trained model
model_path = os.path.join(new_model_dir, 'classification_model_full_data.h5')
model = load_model(model_path)

# 2. Load and filter data
data_test_dir = os.path.join(new_model_dir, 'data/')
original_data_path = os.path.join(data_test_dir, 'all_data_with_SI.csv')
original_data_df = pd.read_csv(original_data_path)

merged_df = original_data_df[original_data_df['SaSaRate'] >= 0.1].copy()

# 3. Predict
X_test = merged_df[[f'embedding_{i}' for i in range(320)]].to_numpy()
predictions = model.predict(X_test)
merged_df['Predicted_Status'] = predictions

# 4. Clean and normalize
merged_df['Predicted_Status'] = pd.to_numeric(merged_df['Predicted_Status'], errors='coerce')

def normalize_group(x):
    if x.isnull().all() or (x.max() == x.min()):
        return np.nan
    else:
        return (x - x.min()) / (x.max() - x.min())

merged_df['Normalized_Predicted_Status'] = merged_df.groupby('name')['Predicted_Status'].transform(normalize_group)

# 5. Drop invalid rows
merged_df = merged_df.replace([np.inf, -np.inf], np.nan)
merged_df = merged_df.dropna(subset=['Normalized_Predicted_Status', 'Predicted_Status', 'Status'])
merged_df = merged_df[merged_df['SaSaRate'] >= 0.1].copy()

# 6. Classification metrics
y_true = merged_df['Status'].replace({'Interacting': 1, 'Non-Interacting': 0}).astype(int)
y_pred = (merged_df['Predicted_Status'] >= 0.5).astype(int)
y_scores = merged_df['Normalized_Predicted_Status']

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, zero_division=0)
rec = recall_score(y_true, y_pred, zero_division=0)
f1 = f1_score(y_true, y_pred, zero_division=0)
auc_val = roc_auc_score(y_true, y_scores)

print("Classification Report:")
print(classification_report(y_true, y_pred))
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"AUC:       {auc_val:.4f}")

# 7. Confusion matrix
conf_matrix = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 6))
plt.imshow(conf_matrix, cmap="Blues", interpolation="nearest")
plt.title(f"Confusion Matrix")
plt.colorbar()
plt.xticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.yticks([0, 1], ['Non-Interacting', 'Interacting'])
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
for i in range(2):
    for j in range(2):
        plt.text(j, i, conf_matrix[i, j], ha="center", va="center", color="red")
plt.show()

# 8. Extra metrics
fpr, tpr, _ = roc_curve(y_true, y_pred)
auc_from_curve = compute_auc(fpr, tpr)
print(f'ROC-AUC from curve: {auc_from_curve:.2f}')

y_pred_binary = (merged_df['Normalized_Predicted_Status'] >= 0.5).astype(int)
balanced_acc = balanced_accuracy_score(y_true, y_pred_binary)
print(f'Balanced Accuracy (BAC): {balanced_acc:.2f}')

mcc_score = matthews_corrcoef(y_true, y_pred_binary)
print(f'MCC: {mcc_score:.2f}')

pr_precision, pr_recall, _ = precision_recall_curve(y_true, y_pred_binary)
pr_auc_score = compute_auc(pr_recall, pr_precision)
print(f'PR-AUC: {pr_auc_score:.2f}')

# 9. Nantigens and Fpredicted
num_unique_antigens = merged_df['name'].nunique()
print(f'Nantigens: {num_unique_antigens}')

fraction_predicted = np.mean(y_pred_binary) * 100
print(f'Fpredicted: {fraction_predicted:.2f}%')

print(
auc_val,
f1,
mcc_score,
pr_auc_score,
balanced_acc,
num_unique_antigens,
fraction_predicted
)